# Notebook 01 — YOLOv11 Football Detection Validation

**Day 1 goal:** Answer one question before writing any architecture:
> *Can YOLO reliably detect football entities on our footage, and how fast is it on this hardware?*

No FootballDetector class. No abstractions. Just raw YOLO -> frames -> measurements.

## Cell 1 — Imports & Config

In [ ]:
import sys
if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except AttributeError:
        pass  # Jupyter kernel OutStream doesn't support reconfigure

import cv2
import numpy as np
import time
import random
import io
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, Image as IPImage

from ultralytics import YOLO

def show_fig(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    display(IPImage(data=buf.read()))
    plt.close(fig)

ROOT        = Path('..').resolve()
CLIPS_DIR   = ROOT / 'data' / 'test_clips'
WEIGHTS_DIR = ROOT / 'weights'

PREFERRED_ORDER = [
    'tactical_playlist_1.mp4',
    'tactical_playlist_2.mp4',
    'tactical_playlist_3.mp4',
    'psg_newcastle_tactical.mp4',
    'mancity_chelsea_tactical.mp4',
    'arsenal_newcastle_highlights.mp4',
    'pl_highlights_1.mp4',
    'pl_highlights_2.mp4',
    'sample.mp4',
]

CLIP_PATH = None
for name in PREFERRED_ORDER:
    candidate = CLIPS_DIR / name
    if candidate.exists() and candidate.stat().st_size > 500_000:
        CLIP_PATH = candidate
        break
if CLIP_PATH is None:
    available = list(CLIPS_DIR.glob('*.mp4'))
    if available:
        CLIP_PATH = max(available, key=lambda p: p.stat().st_size)

assert CLIP_PATH is not None, f'No clips found in {CLIPS_DIR}'
print(f'Clip  : {CLIP_PATH.name}  ({CLIP_PATH.stat().st_size / 1e6:.1f} MB)')

CONF_THRESHOLD = 0.35
IMG_SIZE       = 640
print(f'Config: conf={CONF_THRESHOLD}, imgsz={IMG_SIZE}')
print('Imports OK')

## Cell 2 — Video Metadata

In [ ]:
cap = cv2.VideoCapture(str(CLIP_PATH))
assert cap.isOpened(), f'Could not open {CLIP_PATH}'

W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS      = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
DURATION = N_FRAMES / FPS if FPS > 0 else 0
cap.release()

print('=' * 45)
print(f'  Clip        : {CLIP_PATH.name}')
print(f'  Resolution  : {W} x {H}')
print(f'  FPS         : {FPS:.2f}')
print(f'  Frame count : {N_FRAMES:,}')
print(f'  Duration    : {DURATION:.1f}s  ({DURATION/60:.2f} min)')
print('=' * 45)

frames_60s   = int(FPS * 60)
detect_calls = frames_60s // 3
print(f'\n  Frames in 60s clip    : {frames_60s}')
print(f'  Detection calls (div3): {detect_calls}')
print(f'  At 100ms/call         : {detect_calls * 0.1:.0f}s total')
print(f'  At 25ms/call (OV)     : {detect_calls * 0.025:.0f}s total')

## Cell 3 — Extract Representative Frames
First, middle, and random frames from the clip.

In [ ]:
def read_frame(path, idx):
    cap = cv2.VideoCapture(str(path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise RuntimeError(f'Cannot read frame {idx}')
    return frame

random.seed(42)
idx_first  = min(10, N_FRAMES - 1)
idx_middle = N_FRAMES // 2
idx_random = random.randint(10, max(11, N_FRAMES - 1))

frame_first  = read_frame(CLIP_PATH, idx_first)
frame_middle = read_frame(CLIP_PATH, idx_middle)
frame_random = read_frame(CLIP_PATH, idx_random)

print(f'First  : #{idx_first}  ({idx_first/FPS:.1f}s)')
print(f'Middle : #{idx_middle} ({idx_middle/FPS:.1f}s)')
print(f'Random : #{idx_random} ({idx_random/FPS:.1f}s)')
print(f'Shape  : {frame_first.shape}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Three Sample Frames -- {CLIP_PATH.name}', fontsize=12, fontweight='bold')
for ax, frame, lbl in zip(
    axes,
    [frame_first, frame_middle, frame_random],
    [f'First (#{idx_first}, {idx_first/FPS:.1f}s)',
     f'Middle (#{idx_middle}, {idx_middle/FPS:.1f}s)',
     f'Random (#{idx_random}, {idx_random/FPS:.1f}s)']):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(lbl, fontsize=9)
    ax.axis('off')
plt.tight_layout()
show_fig(fig)

## Cell 4 — Load YOLOv11n
Base COCO model: class 0=person (all players+referee), class 32=sports ball.
After fine-tuning on Roboflow football dataset, swap in the fine-tuned weights.

In [ ]:
t0 = time.perf_counter()
model = YOLO('yolo11n.pt')
load_ms = (time.perf_counter() - t0) * 1000

CLASS_NAMES_RAW   = model.names
FOOTBALL_CLASSES  = {'player', 'goalkeeper', 'referee', 'ball'}
IS_FOOTBALL_MODEL = bool(FOOTBALL_CLASSES & set(CLASS_NAMES_RAW.values()))

if IS_FOOTBALL_MODEL:
    print('Detected: FOOTBALL fine-tuned model')
    name_to_id = {v: k for k, v in CLASS_NAMES_RAW.items()}
    TARGET_CLASSES = {name_to_id.get(n, -1): n
                      for n in ('player', 'goalkeeper', 'referee', 'ball')}
else:
    print('Detected: Base COCO model (not yet fine-tuned for football)')
    print('  -> class 0  (person)      = all players + referee combined')
    print('  -> class 32 (sports ball) = football')
    TARGET_CLASSES = {0: 'person', 32: 'sports ball'}

CLASS_COLORS = {
    'player':      (220,  50,   0),
    'goalkeeper':  (255, 165,   0),
    'referee':     ( 30,  30,  30),
    'ball':        (  0, 200, 200),
    'person':      (220,  50,   0),
    'sports ball': (  0, 200, 200),
}

print(f'\nModel loaded in {load_ms:.0f}ms')
print(f'Task          : {model.task}')
print(f'Total classes : {len(CLASS_NAMES_RAW)}')
print(f'Tracking      : {list(TARGET_CLASSES.values())}')

## Cell 5 — Run Detection on Three Frames

In [ ]:
def run_detection(frame, conf=CONF_THRESHOLD, imgsz=IMG_SIZE):
    t0 = time.perf_counter()
    results = model(frame, conf=conf, imgsz=imgsz, verbose=False)
    ms = (time.perf_counter() - t0) * 1000
    return results[0], ms

def count_classes(results):
    counts = {name: 0 for name in TARGET_CLASSES.values()}
    boxes = results.boxes
    if boxes is None or len(boxes) == 0:
        return counts
    for cls_id in boxes.cls.numpy().astype(int):
        name = TARGET_CLASSES.get(cls_id)
        if name:
            counts[name] += 1
    return counts

# Warm-up
_ = model(frame_first, conf=CONF_THRESHOLD, imgsz=IMG_SIZE, verbose=False)

res_first,  ms_first  = run_detection(frame_first)
res_middle, ms_middle = run_detection(frame_middle)
res_random, ms_random = run_detection(frame_random)

print(f'{"Frame":<10} {"ms":>6}  {"total":>6}  breakdown')
print('-' * 55)
for label, res, ms in [('first',  res_first,  ms_first),
                        ('middle', res_middle, ms_middle),
                        ('random', res_random, ms_random)]:
    n   = len(res.boxes) if res.boxes else 0
    cnts = count_classes(res)
    bd   = '  '.join(f'{nm}:{c}' for nm, c in cnts.items() if c > 0)
    print(f'{label:<10} {ms:>5.0f}ms  {n:>6}  {bd or "none"}')

## Cell 6 — Visualise Detections

In [ ]:
def draw_detections(frame, results, target_classes, class_colors):
    ann = frame.copy()
    boxes = results.boxes
    if boxes is None or len(boxes) == 0:
        return ann
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i].item())
        conf   = float(boxes.conf[i].item())
        if cls_id not in target_classes:
            continue
        name  = target_classes[cls_id]
        color = class_colors.get(name, (128, 128, 128))
        bgr   = color[::-1]
        x1, y1, x2, y2 = boxes.xyxy[i].numpy().astype(int)
        cv2.rectangle(ann, (x1, y1), (x2, y2), bgr, 2)
        label = f'{name} {conf:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        cv2.rectangle(ann, (x1, y1 - th - 6), (x1 + tw + 4, y1), bgr, -1)
        cv2.putText(ann, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return ann

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle(f'YOLOv11n Detections -- conf={CONF_THRESHOLD}, imgsz={IMG_SIZE}',
             fontsize=12, fontweight='bold')
for ax, (frame, res, ms, title) in zip(axes, [
    (frame_first,  res_first,  ms_first,  f'First  (#{idx_first})'),
    (frame_middle, res_middle, ms_middle, f'Middle (#{idx_middle})'),
    (frame_random, res_random, ms_random, f'Random (#{idx_random})'),
]):
    ann = draw_detections(frame, res, TARGET_CLASSES, CLASS_COLORS)
    n   = len(res.boxes) if res.boxes else 0
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{title}\n{n} detections | {ms:.0f}ms', fontsize=9)
    ax.axis('off')

legend_patches = [
    mpatches.Patch(color=tuple(c/255 for c in col), label=name)
    for name, col in CLASS_COLORS.items() if name in TARGET_CLASSES.values()
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=len(legend_patches), fontsize=10, framealpha=0.9)
plt.tight_layout(rect=[0, 0.06, 1, 1])
show_fig(fig)

## Cell 7 — Detection Counts Per Class

In [ ]:
class_list = list(dict.fromkeys(TARGET_CLASSES.values()))
header = f'{"Frame":<10}' + ''.join(f'{n:>14}' for n in class_list) + f'{"TOTAL":>10}'
print(f'Detection counts per class (conf={CONF_THRESHOLD}):')
print(header)
print('-' * len(header))
for label, res in [('First', res_first), ('Middle', res_middle), ('Random', res_random)]:
    cnts  = count_classes(res)
    total = sum(cnts.values())
    print(f'{label:<10}' + ''.join(f'{cnts.get(n, 0):>14}' for n in class_list) + f'{total:>10}')

print()
avg_persons = sum(
    count_classes(r).get('person', 0) + count_classes(r).get('player', 0)
    for r in [res_first, res_middle, res_random]
) / 3
print(f'Average player/person detections: {avg_persons:.1f}')
if avg_persons >= 15:
    print('  -> GOOD -- most players visible on this clip')
elif avg_persons >= 8:
    print('  -> OK -- partial detection, check conf or clip quality')
elif avg_persons >= 3:
    print('  -> POOR -- try conf=0.25 or a wider-angle clip')
else:
    print('  -> FAIL -- broadcast close-ups may not show many players at once')

## Cell 8 — 100-Frame Inference Benchmark

In [ ]:
N_BENCH = 100
print(f'Running benchmark: {N_BENCH} consecutive frames...')

cap = cv2.VideoCapture(str(CLIP_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 10)

latencies        = []
bench_det_counts = []

for i in range(N_BENCH):
    ret, frame = cap.read()
    if not ret:
        print(f'  Clip ended at frame {i}')
        break
    t0  = time.perf_counter()
    res = model(frame, conf=CONF_THRESHOLD, imgsz=IMG_SIZE, verbose=False)
    ms  = (time.perf_counter() - t0) * 1000
    latencies.append(ms)
    n = sum(1 for c in (res[0].boxes.cls.numpy().astype(int)
                        if res[0].boxes and len(res[0].boxes) > 0 else [])
            if c in TARGET_CLASSES)
    bench_det_counts.append(n)

cap.release()

latencies        = np.array(latencies)
bench_det_counts = np.array(bench_det_counts)

print(f'\nBenchmark results ({len(latencies)} frames, imgsz={IMG_SIZE})')
print(f'  Mean latency    : {latencies.mean():.1f} ms/frame')
print(f'  Median latency  : {np.median(latencies):.1f} ms/frame')
print(f'  p95 latency     : {np.percentile(latencies, 95):.1f} ms/frame')
print(f'  Min / Max       : {latencies.min():.1f} / {latencies.max():.1f} ms')
print(f'  Detections/frame: mean={bench_det_counts.mean():.1f}, '
      f'min={bench_det_counts.min()}, max={bench_det_counts.max()}')
print()
print('60s clip estimates (PyTorch CPU):')
for fps_clip in [25, 30]:
    for skip in [1, 3, 5]:
        calls = (fps_clip * 60) // skip
        est   = latencies.mean() * calls / 1000
        print(f'  {fps_clip}fps, skip-{skip}: {calls} calls -> ~{est:.0f}s')

## Cell 9 — Latency Plots

In [ ]:
x = np.arange(len(latencies))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
ax1.plot(x, latencies, color='steelblue', linewidth=0.8, alpha=0.8, label='ms/frame')
ax1.axhline(latencies.mean(), color='red', linewidth=1.5, linestyle='--',
            label=f'Mean {latencies.mean():.0f}ms')
ax1.axhline(np.median(latencies), color='green', linewidth=1.5, linestyle=':',
            label=f'Median {np.median(latencies):.0f}ms')
ax1.fill_between(x, np.percentile(latencies, 25), np.percentile(latencies, 75),
                 alpha=0.15, color='steelblue', label='IQR')
ax1.set_xlabel('Frame #')
ax1.set_ylabel('Inference time (ms)')
ax1.set_title(f'YOLOv11n Latency -- {CLIP_PATH.name}')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

ax2.plot(x, bench_det_counts, color='darkorange', linewidth=0.8, alpha=0.8)
ax2.axhline(bench_det_counts.mean(), color='red', linewidth=1.5, linestyle='--',
            label=f'Mean {bench_det_counts.mean():.1f}')
ax2.fill_between(x, 0, bench_det_counts, alpha=0.15, color='darkorange')
ax2.set_xlabel('Frame #')
ax2.set_ylabel('Detections per frame')
ax2.set_title('Target-Class Detections per Frame')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
show_fig(fig)

fig2, ax = plt.subplots(figsize=(8, 4))
ax.hist(latencies, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(latencies.mean(), color='red', linewidth=2, linestyle='--',
           label=f'Mean {latencies.mean():.0f}ms')
ax.axvline(np.percentile(latencies, 95), color='purple', linewidth=2, linestyle=':',
           label=f'p95 {np.percentile(latencies, 95):.0f}ms')
ax.set_xlabel('Inference time (ms)')
ax.set_ylabel('Frame count')
ax.set_title('Latency Distribution')
ax.legend()
plt.tight_layout()
show_fig(fig2)

## Cell 10 — Confidence Threshold Sweep

In [ ]:
conf_values   = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
sweep_results = []

for conf in conf_values:
    t0  = time.perf_counter()
    res = model(frame_middle, conf=conf, imgsz=IMG_SIZE, verbose=False)[0]
    ms  = (time.perf_counter() - t0) * 1000
    cnts = count_classes(res)
    sweep_results.append((conf, len(res.boxes) if res.boxes else 0, cnts, ms))

col_names = list(dict.fromkeys(TARGET_CLASSES.values()))
header = f'{"conf":<6}  {"total":>6}  {"ms":>6}  ' + ''.join(f'{n:>13}' for n in col_names)
print(f'Confidence sweep (middle frame, imgsz={IMG_SIZE}):')
print(header)
print('-' * len(header))
for conf, total, cnts, ms in sweep_results:
    marker = '  <-- current' if abs(conf - CONF_THRESHOLD) < 0.001 else ''
    row = f'{conf:<6.2f}  {total:>6}  {ms:>5.0f}ms  ' + ''.join(f'{cnts.get(n, 0):>13}' for n in col_names)
    print(row + marker)

show_confs = [0.20, 0.30, 0.35, 0.45]
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Confidence Sweep -- Middle Frame', fontsize=12, fontweight='bold')
for ax, conf in zip(axes, show_confs):
    res = model(frame_middle, conf=conf, imgsz=IMG_SIZE, verbose=False)[0]
    ann = draw_detections(frame_middle, res, TARGET_CLASSES, CLASS_COLORS)
    n   = len(res.boxes) if res.boxes else 0
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'conf={conf:.2f} -> {n} det', fontsize=10)
    ax.axis('off')
plt.tight_layout()
show_fig(fig)

## Cell 11 — Detection Quality Evaluation
Ball reliability, missed players, detection stability over 50 frames.

In [ ]:
ball_class_ids = [k for k, v in TARGET_CLASSES.items() if 'ball' in v.lower()]

cap = cv2.VideoCapture(str(CLIP_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 10)

N_QUALITY   = min(50, N_FRAMES - 10)
ball_found  = []
person_cnts = []

for _ in range(N_QUALITY):
    ret, frame = cap.read()
    if not ret: break
    res = model(frame, conf=CONF_THRESHOLD, imgsz=IMG_SIZE, verbose=False)[0]
    cls_ids = (res.boxes.cls.numpy().astype(int)
               if res.boxes and len(res.boxes) > 0 else np.array([]))
    ball_found.append(any(c in ball_class_ids for c in cls_ids))
    person_cnts.append(sum(1 for c in cls_ids
                           if TARGET_CLASSES.get(c, '').lower() in
                           ('person', 'player', 'goalkeeper', 'referee')))

cap.release()

ball_pct    = np.mean(ball_found) * 100
person_mean = np.mean(person_cnts)
person_std  = np.std(person_cnts)
miss_thresh = max(2, person_mean * 0.5)
miss_cnt    = sum(1 for c in person_cnts if c < miss_thresh)

print('=' * 50)
print('  DETECTION QUALITY REPORT')
print('=' * 50)
print(f'  Frames sampled   : {len(person_cnts)}')
print(f'  Conf threshold   : {CONF_THRESHOLD}')
print(f'\n  Player/Person:')
print(f'    Mean/frame     : {person_mean:.1f}')
print(f'    Std deviation  : {person_std:.1f}')
print(f'    Low-det frames : {miss_cnt}/{len(person_cnts)} '
      f'({miss_cnt/len(person_cnts)*100:.0f}%)')
print(f'\n  Ball:')
print(f'    Found in {sum(ball_found)}/{len(ball_found)} frames ({ball_pct:.0f}%)')
if   ball_pct >= 60: print('    Reliability: GOOD')
elif ball_pct >= 30: print('    Reliability: MODERATE (often occluded or small)')
else:                print('    Reliability: POOR -- fine-tuned model needed')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(person_cnts, color='steelblue', linewidth=1)
ax1.axhline(person_mean, color='red', linestyle='--', linewidth=1.5, label=f'Mean {person_mean:.1f}')
ax1.axhline(miss_thresh, color='orange', linestyle=':', linewidth=1.5, label=f'Threshold {miss_thresh:.0f}')
ax1.fill_between(range(len(person_cnts)), miss_thresh, person_cnts,
                 where=[c < miss_thresh for c in person_cnts],
                 color='red', alpha=0.3, label='Low-det')
ax1.set_xlabel('Frame #')
ax1.set_ylabel('Persons detected')
ax1.set_title('Detection Stability')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(['Found', 'Missed'],
        [sum(ball_found), len(ball_found) - sum(ball_found)],
        color=['#2ecc71', '#e74c3c'])
ax2.set_ylabel('Frames')
ax2.set_title(f'Ball Detection ({ball_pct:.0f}%, {len(ball_found)} frames)')
ax2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
show_fig(fig)

## Cell 12 — Frame-Skip Strategy Analysis
Detect every N frames, carry bboxes forward. Measure accuracy vs speed trade-off.

In [ ]:
N_SKIP = min(60, N_FRAMES - 10)
cap = cv2.VideoCapture(str(CLIP_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, 10)

skip_counts = []
skip_lats   = []
for _ in range(N_SKIP):
    ret, frame = cap.read()
    if not ret: break
    t0  = time.perf_counter()
    res = model(frame, conf=CONF_THRESHOLD, imgsz=IMG_SIZE, verbose=False)[0]
    skip_lats.append((time.perf_counter() - t0) * 1000)
    skip_counts.append(len(res.boxes) if res.boxes else 0)
cap.release()

skip_counts = np.array(skip_counts)
skip_lats   = np.array(skip_lats)
deltas      = np.abs(np.diff(skip_counts))

print(f'Frame-skip analysis ({len(skip_counts)} frames):')
print(f'  Mean frame delta : {deltas.mean():.2f}')
print(f'  Max frame delta  : {deltas.max()}')
print()
for skip in [2, 3, 5]:
    sim = np.array([
        skip_counts[min((i // skip) * skip, len(skip_counts)-1)]
        for i in range(len(skip_counts))
    ])
    error = np.abs(skip_counts - sim).mean()
    est_s = skip_lats.mean() * (len(skip_counts) // skip) / 1000
    print(f'  skip-{skip}: error={error:.2f} det, speedup={skip}x, ~{est_s:.1f}s for {len(skip_counts)} frames')

print()
if   deltas.mean() < 2.0: print('  Verdict: LOW change -> skip-3 is safe')
elif deltas.mean() < 4.0: print('  Verdict: MODERATE -> skip-3 acceptable')
else:                      print('  Verdict: HIGH change -> use skip-2 max')

sim3 = np.array([
    skip_counts[min((i // 3) * 3, len(skip_counts)-1)]
    for i in range(len(skip_counts))
])
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(skip_counts, label='Every frame (truth)', color='steelblue', linewidth=1)
ax.plot(sim3, label='skip-3 (carry-forward)', color='red', linewidth=1, linestyle='--', alpha=0.8)
ax.set_xlabel('Frame #')
ax.set_ylabel('Detections')
ax.set_title('Every Frame vs skip-3 Strategy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
show_fig(fig)

## Cell 13 — OpenVINO Speed Benchmark
Uses the `openvino` Python API directly (avoids Ultralytics YOLO wrapper naming conventions).
OV export at `weights/yolo11n_openvino_verify/` outputs shape `[1, 84, 8400]` — YOLO detection format.

In [ ]:
OV_PATH   = WEIGHTS_DIR / 'yolo11n_openvino_verify'
xml_files = list(OV_PATH.glob('*.xml')) if OV_PATH.exists() else []

if not xml_files:
    print(f'OpenVINO model not found at {OV_PATH}')
    print('Run day0_verify.py first (Check 5 exports it automatically)')
else:
    try:
        import openvino as ov

        core     = ov.Core()
        ov_model = core.read_model(str(xml_files[0]))
        compiled = core.compile_model(ov_model, 'CPU')

        print(f'Loaded: {xml_files[0].name}')
        print(f'Input  : {compiled.input(0).shape}')
        print(f'Output : {compiled.output(0).shape}')

        def preprocess_ov(frame, imgsz=640):
            img = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (imgsz, imgsz))
            img = img.astype(np.float32) / 255.0
            return np.expand_dims(np.transpose(img, (2, 0, 1)), 0)

        inp = preprocess_ov(frame_middle)

        # Warm-up
        for _ in range(3):
            _ = compiled([inp])

        N_OV    = 50
        ov_lats = []
        for _ in range(N_OV):
            t0 = time.perf_counter()
            _  = compiled([inp])
            ov_lats.append((time.perf_counter() - t0) * 1000)

        ov_lats  = np.array(ov_lats)
        cpu_mean = latencies.mean()
        ov_mean  = ov_lats.mean()
        speedup  = cpu_mean / ov_mean if ov_mean > 0 else 1.0

        print(f'\nSpeed comparison ({N_OV} runs):')
        print(f'  PyTorch CPU  : {cpu_mean:.0f} ms/frame')
        print(f'  OpenVINO CPU : {ov_mean:.0f} ms/frame')
        print(f'  Speedup      : {speedup:.2f}x')

        dc60 = int(FPS * 60) // 3
        print(f'\n  60s clip, every-3rd-frame ({dc60} calls):')
        print(f'    PyTorch  : ~{cpu_mean * dc60 / 1000:.0f}s')
        print(f'    OpenVINO : ~{ov_mean  * dc60 / 1000:.0f}s')

        if   ov_mean < 50:  print(f'\n  OpenVINO: FAST (<50ms) -> use for production')
        elif ov_mean < 100: print(f'\n  OpenVINO: OK ({ov_mean:.0f}ms) -> better than PyTorch')
        else:               print(f'\n  OpenVINO: SLOW ({ov_mean:.0f}ms) -> check Arc GPU passthrough')

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(latencies[:N_OV], bins=20, alpha=0.6, color='steelblue',
                label=f'PyTorch CPU (mean {cpu_mean:.0f}ms)')
        ax.hist(ov_lats,          bins=20, alpha=0.6, color='green',
                label=f'OpenVINO CPU (mean {ov_mean:.0f}ms)')
        ax.set_xlabel('Inference time (ms)')
        ax.set_ylabel('Count')
        ax.set_title('PyTorch vs OpenVINO Latency Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        show_fig(fig)

    except Exception as e:
        print(f'OpenVINO benchmark failed: {type(e).__name__}: {e}')

## Cell 14 — Final Recommendations

In [ ]:
sweep_persons = [
    (conf, sum(cnts.get(n, 0) for n in ('person', 'player', 'goalkeeper', 'referee')))
    for conf, total, cnts, ms in sweep_results
    if 0.20 <= conf <= 0.45
]
best_conf  = max(sweep_persons, key=lambda x: x[1])[0] if sweep_persons else 0.35
avg_delta  = np.abs(np.diff(skip_counts)).mean() if len(skip_counts) > 1 else 3
rec_skip   = 3 if avg_delta < 2 else (2 if avg_delta < 4 else 1)

print('=' * 55)
print('  FINAL RECOMMENDATIONS')
print('=' * 55)
print(f'\n  DETECTION_CONF_THRESHOLD = {best_conf}')
print(f'  DETECTION_IMG_SIZE       = 640')
print(f'  DETECT_EVERY_N_FRAMES    = {rec_skip}')
print(f'\n  Reasoning:')
print(f'    conf={best_conf}: peak person-count in 0.20-0.45 sweep')
print(f'    imgsz=640: standard, OpenVINO-compatible')
print(f'    skip={rec_skip}: mean frame delta was {avg_delta:.1f} detections')
print(f'\n  Model:')
if IS_FOOTBALL_MODEL:
    print('    Fine-tuned -- goalkeeper/referee/ball classes available')
else:
    print('    Base COCO -- person covers players+referee combined')
    print('    Next: fine-tune on Roboflow football dataset (Colab GPU)')
print(f'\n  Hardware:')
print(f'    PyTorch CPU mean : {latencies.mean():.0f} ms/frame')
if   latencies.mean() < 80:  print('    Status: FAST -- CPU fine for development')
elif latencies.mean() < 150: print('    Status: OK -- use OpenVINO for production')
else:                        print('    Status: SLOW -- OpenVINO required; consider imgsz=480')
print(f'\n  Next steps:')
print('    1. Set values above in gaffer/config.py')
print('    2. Build gaffer/detection/detector.py (FootballDetector class)')
print('    3. Build gaffer/detection/team_assigner.py (K-Means jerseys)')
print('    4. Open notebooks/02_tracking_test.ipynb')
if not IS_FOOTBALL_MODEL:
    print('    5. Fine-tune yolo11n on Roboflow football dataset')
print('=' * 55)